# 01 — Preprocess the HCRL Car-Hacking dataset

What this notebook does:
1. Load the four attack CSV files (DLC-variable rows; the last field is the label R/T).
2. Load the normal-run text log (which has a slightly different format).
3. Sanity-check the label and source distribution.
4. Engineer features (hex → int for the CAN ID and the eight data bytes; label T → 1, R → 0).
5. Save the combined dataset to CSV.
6. Build a 4-client non-IID partition for the CFL experiments in 02.
7. Print a short summary.

> Select the **Python (axi)** kernel.

In [1]:
# imports + config
from pathlib import Path
import re
import pandas as pd
import numpy as np

DATA_DIR = Path('/Users/lucia/Dropbox/USYD/Semester1_2026/AXI/9) Car-Hacking Dataset')
OUT_DIR  = Path('/Users/lucia/Dropbox/USYD/Semester1_2026/AXI/data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PER_ATTACK = 50_000    # random sample per attack file (kept small for speed; scale up later if needed)
SAMPLE_NORMAL     = 50_000
RANDOM_SEED       = 42

print(f'DATA_DIR = {DATA_DIR}')
print(f'OUT_DIR  = {OUT_DIR}')
print(f'SAMPLE_PER_ATTACK = {SAMPLE_PER_ATTACK:,}')
print(f'SAMPLE_NORMAL     = {SAMPLE_NORMAL:,}')

DATA_DIR = /Users/lucia/Dropbox/USYD/Semester1_2026/AXI/9) Car-Hacking Dataset
OUT_DIR  = /Users/lucia/Dropbox/USYD/Semester1_2026/AXI/data/processed
SAMPLE_PER_ATTACK = 50,000
SAMPLE_NORMAL     = 50,000


## 1. Load attack CSVs (DLC-variable rows; the last field is the label)

In [2]:
def random_sample_lines(path, n, seed=RANDOM_SEED):
    '''Return n random lines from a (large) file, memory-efficiently.'''
    with open(path, 'rb') as f:
        total = sum(1 for _ in f)
    rng = np.random.default_rng(seed)
    n = min(n, total)
    keep = set(rng.choice(total, size=n, replace=False).tolist())
    sampled = []
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if i in keep:
                sampled.append(line.rstrip('\n'))
    return sampled, total

def parse_attack_rows(lines):
    rows = []
    for line in lines:
        parts = line.split(',')
        if len(parts) < 4:
            continue
        try:
            ts = float(parts[0])
            can_id_hex = parts[1].strip()
            dlc = int(parts[2])
            label = parts[-1].strip()           # 'R' (normal) or 'T' (injected)
            data_hex = parts[3:3+dlc]
            data_hex = (data_hex + ['00']*8)[:8]  # pad to a fixed 8 bytes
            rows.append((ts, can_id_hex, dlc, *data_hex, label))
        except (ValueError, IndexError):
            continue
    cols = ['ts','can_id_hex','dlc'] + [f'd{i}' for i in range(8)] + ['flag']
    return pd.DataFrame(rows, columns=cols)

def load_attack(name, sample_n=SAMPLE_PER_ATTACK):
    path = DATA_DIR / f'{name}_dataset.csv'
    lines, total = random_sample_lines(path, sample_n)
    df = parse_attack_rows(lines)
    df['source'] = name
    T = int((df.flag=='T').sum()); R = int((df.flag=='R').sum())
    print(f'  {name:6s} total={total:>10,}  sampled={len(df):>7,}  T={T:>6,}  R={R:>7,}')
    return df

print('Loading the four attack files...')
attack_df = pd.concat([load_attack(nm) for nm in ['DoS','Fuzzy','RPM','gear']], ignore_index=True)
print(f'\nattack rows total: {attack_df.shape}')
attack_df.head()

Loading the four attack files...


  DoS    total= 3,665,771  sampled= 50,000  T= 7,992  R= 42,008


  Fuzzy  total= 3,838,860  sampled= 50,000  T= 6,554  R= 43,446


  RPM    total= 4,621,702  sampled= 50,000  T= 7,136  R= 42,864


  gear   total= 4,443,142  sampled= 50,000  T= 6,813  R= 43,187

attack rows total: (200000, 13)


,ts,can_id_hex,dlc,d0,d1,d2,d3,d4,d5,d6,d7,flag,source
0,1.478198e+09,02a0,8,64,00,9a,1d,97,02,bd,00,R,DoS
1,1.478198e+09,0140,8,00,00,00,00,08,22,2b,a3,R,DoS
2,1.478198e+09,0131,8,12,80,00,00,30,7f,0d,88,R,DoS
3,1.478198e+09,0131,8,05,80,00,00,36,7f,0e,9d,R,DoS
4,1.478198e+09,0153,8,00,21,10,ff,00,ff,00,00,R,DoS


## 2. Load the normal-run text log

Format: `Timestamp: <float>    ID: <hex>    000    DLC: <n>    <hex> x n` (no label; all rows are normal).

In [3]:
NORMAL_PATH = DATA_DIR / 'normal_run_data' / 'normal_run_data.txt'
PAT = re.compile(r'Timestamp:\s*(\S+)\s+ID:\s*(\S+)\s+\S+\s+DLC:\s*(\d+)\s+(.*)')

def parse_normal_lines(lines):
    rows = []
    for line in lines:
        m = PAT.match(line)
        if not m:
            continue
        ts = float(m.group(1))
        can_id_hex = m.group(2).strip()
        dlc = int(m.group(3))
        data_hex = m.group(4).strip().split()
        data_hex = (data_hex + ['00']*8)[:8]
        rows.append((ts, can_id_hex, dlc, *data_hex, 'R'))   # all normal
    cols = ['ts','can_id_hex','dlc'] + [f'd{i}' for i in range(8)] + ['flag']
    return pd.DataFrame(rows, columns=cols)

print('Loading the normal-run data...')
lines, total = random_sample_lines(NORMAL_PATH, SAMPLE_NORMAL)
normal_df = parse_normal_lines(lines)
normal_df['source'] = 'Normal'
print(f'  normal total={total:>10,}  sampled={len(normal_df):>7,}')
normal_df.head()

Loading the normal-run data...


  normal total=   988,872  sampled= 50,000


,ts,can_id_hex,dlc,d0,d1,d2,d3,d4,d5,d6,d7,flag,source
0,1.479121e+09,0350,8,05,28,84,66,6d,00,00,a2,R,Normal
1,1.479121e+09,0430,8,00,00,00,00,00,00,00,00,R,Normal
2,1.479121e+09,02a0,8,60,00,6b,1d,01,04,dd,00,R,Normal
3,1.479121e+09,0370,8,00,20,00,00,00,00,00,00,R,Normal
4,1.479121e+09,0430,8,00,00,00,00,00,00,00,00,R,Normal


## 3. Sanity check: label and source distribution

In [4]:
all_df = pd.concat([attack_df, normal_df], ignore_index=True)
print('source counts:'); print(all_df['source'].value_counts(), '\n')
print('flag counts (overall):'); print(all_df['flag'].value_counts(), '\n')
print('source x flag crosstab:')
print(pd.crosstab(all_df['source'], all_df['flag']))

source counts:
source
DoS       50000
Fuzzy     50000
RPM       50000
gear      50000
Normal    50000
Name: count, dtype: int64 

flag counts (overall):
flag
R    221505
T     28495
Name: count, dtype: int64 

source x flag crosstab:
flag        R     T
source             
DoS     42008  7992
Fuzzy   43446  6554
Normal  50000     0
RPM     42864  7136
gear    43187  6813


## 4. Feature engineering

- `can_id` : hex → int
- `b0..b7` : data bytes hex → int (0–255)
- `label`  : T → 1 (attack), R → 0 (normal)

In [5]:
def hex_to_int(s):
    try: return int(str(s), 16)
    except Exception: return 0

all_df['can_id'] = all_df['can_id_hex'].apply(hex_to_int)
for i in range(8):
    all_df[f'b{i}'] = all_df[f'd{i}'].apply(hex_to_int)
all_df['label'] = (all_df['flag']=='T').astype(int)

feat_cols = ['can_id','dlc'] + [f'b{i}' for i in range(8)]
print('feature columns:', feat_cols)
print(f'shape: {all_df.shape}  /  attack (label=1) ratio: {all_df.label.mean():.3f}')
all_df[feat_cols + ['label','source']].head()

feature columns: ['can_id', 'dlc', 'b0', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7']
shape: (250000, 23)  /  attack (label=1) ratio: 0.114


,can_id,dlc,b0,b1,b2,b3,b4,b5,b6,b7,label,source
0,672,8,100,0,154,29,151,2,189,0,0,DoS
1,320,8,0,0,0,0,8,34,43,163,0,DoS
2,305,8,18,128,0,0,48,127,13,136,0,DoS
3,305,8,5,128,0,0,54,127,14,157,0,DoS
4,339,8,0,33,16,255,0,255,0,0,0,DoS


## 5. Save the combined dataset (CSV)

In [6]:
keep_cols = ['ts','source'] + feat_cols + ['label']
processed = all_df[keep_cols].copy()
out_path = OUT_DIR / 'can_combined.csv'
processed.to_csv(out_path, index=False)
print(f'saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)')
print(f'shape: {processed.shape}')

saved: /Users/lucia/Dropbox/USYD/Semester1_2026/AXI/data/processed/can_combined.csv  (13.2 MB)
shape: (250000, 13)


## 6. 4-client non-IID partition for the CFL experiments

Each simulated client (vehicle) holds normal traffic plus mostly **one** attack type, with a small 10% share of the other attacks for realism. This non-IID heterogeneity is what makes the remember/forget dynamics visible in 02.

(The setup is compatible with the standard CFL protocol used by FOT, 2023.)

In [7]:
attack_types = ['DoS','Fuzzy','RPM','gear']
clients = {}

normal_pool  = processed[processed['source']=='Normal'].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
n_chunks     = len(attack_types)
chunk_size   = len(normal_pool) // n_chunks
normal_split = [normal_pool.iloc[i*chunk_size:(i+1)*chunk_size if i<n_chunks-1 else len(normal_pool)]
                for i in range(n_chunks)]

for i, atk in enumerate(attack_types):
    own    = processed[processed['source']==atk]
    others = processed[processed['source'].isin([a for a in attack_types if a!=atk])]
    other_share = others.sample(frac=0.10, random_state=RANDOM_SEED+i)
    df_c = pd.concat([normal_split[i], own, other_share], ignore_index=True) \
              .sample(frac=1, random_state=RANDOM_SEED+i).reset_index(drop=True)
    name = f'client_{i}_{atk}'
    clients[name] = df_c
    src_dist = dict(df_c.source.value_counts())
    print(f'  {name:18s} rows={len(df_c):>6,}  label1={df_c.label.mean():.3f}  source={src_dist}')

client_dir = OUT_DIR / 'clients'
client_dir.mkdir(exist_ok=True)
for name, df in clients.items():
    df.to_csv(client_dir / f'{name}.csv', index=False)
print(f'\nsaved client files under: {client_dir}')

  client_0_DoS       rows=77,500  label1=0.130  source={'DoS': np.int64(50000), 'Normal': np.int64(12500), 'RPM': np.int64(5083), 'gear': np.int64(4979), 'Fuzzy': np.int64(4938)}
  client_1_Fuzzy     rows=77,500  label1=0.112  source={'Fuzzy': np.int64(50000), 'Normal': np.int64(12500), 'gear': np.int64(5031), 'DoS': np.int64(5031), 'RPM': np.int64(4938)}
  client_2_RPM       rows=77,500  label1=0.120  source={'RPM': np.int64(50000), 'Normal': np.int64(12500), 'gear': np.int64(5071), 'DoS': np.int64(4984), 'Fuzzy': np.int64(4945)}
  client_3_gear      rows=77,500  label1=0.116  source={'gear': np.int64(50000), 'Normal': np.int64(12500), 'RPM': np.int64(5059), 'Fuzzy': np.int64(4991), 'DoS': np.int64(4950)}



saved client files under: /Users/lucia/Dropbox/USYD/Semester1_2026/AXI/data/processed/clients


## 7. Summary

In [8]:
print('='*60); print('Preprocessing summary'); print('='*60)
print(f'combined dataset: {processed.shape}')
print(f'  normal (label=0): {int((processed.label==0).sum()):,}')
print(f'  attack (label=1): {int((processed.label==1).sum()):,}')
src_attack = processed[processed.source.isin(attack_types)].source.value_counts().to_dict()
print(f'  attack-type counts: {src_attack}')
print('\nclient partition:')
for name, df in clients.items():
    print(f'  {name:18s}: rows={len(df):>6,}  label1={df.label.mean():.3f}')
print('\nfiles saved:')
print(f'  {out_path}')
print(f'  {client_dir}/<client>.csv  x {len(clients)}')
print('\nDone. Next: 02_fl_baseline')

Preprocessing summary
combined dataset: (250000, 13)
  normal (label=0): 221,505
  attack (label=1): 28,495
  attack-type counts: {'DoS': 50000, 'Fuzzy': 50000, 'RPM': 50000, 'gear': 50000}

client partition:
  client_0_DoS      : rows=77,500  label1=0.130
  client_1_Fuzzy    : rows=77,500  label1=0.112
  client_2_RPM      : rows=77,500  label1=0.120
  client_3_gear     : rows=77,500  label1=0.116

files saved:
  /Users/lucia/Dropbox/USYD/Semester1_2026/AXI/data/processed/can_combined.csv
  /Users/lucia/Dropbox/USYD/Semester1_2026/AXI/data/processed/clients/<client>.csv  x 4

Done. Next: 02_fl_baseline
